# Association Rules Assignment

In [ ]:

# Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.frequent_patterns import apriori, association_rules


In [ ]:

# Load Dataset

df = pd.read_excel('Online retail.xlsx')

print(df.head())
print(df.shape)


In [ ]:

# Dataset Information

print(df.info())

print(df.isnull().sum())


In [ ]:

# Remove Missing Values

df.dropna(inplace=True)

print(df.shape)


In [ ]:

# Remove Duplicate Rows

df.drop_duplicates(inplace=True)

print(df.shape)


In [ ]:

# Top Products

top_products = df['Description'].value_counts().head(10)

print(top_products)


In [ ]:

# Visualization

top_products.plot(kind='bar', figsize=(10,5))

plt.title('Top Selling Products')

plt.show()


In [ ]:

# Basket Analysis

basket = (
    df.groupby(['InvoiceNo', 'Description'])['Quantity']
    .sum()
    .unstack()
    .reset_index()
    .fillna(0)
    .set_index('InvoiceNo')
)

basket = basket.applymap(lambda x: 1 if x > 0 else 0)

print(basket.head())


In [ ]:

# Apply Apriori Algorithm

frequent_itemsets = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

print(frequent_itemsets.head())


In [ ]:

# Generate Association Rules

rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=1
)

print(rules.head())


In [ ]:

# Sort Rules by Confidence

rules = rules.sort_values(
    by='confidence',
    ascending=False
)

print(rules[['antecedents',
             'consequents',
             'support',
             'confidence',
             'lift']].head())


In [ ]:

# Threshold Experimentation

rules_1 = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.3
)

rules_2 = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.5
)

print('Rules with threshold 0.3:', len(rules_1))
print('Rules with threshold 0.5:', len(rules_2))



# Recommendation Function


In [ ]:

def recommend_products(product_name):

    recommendations = rules[
        rules['antecedents'].apply(
            lambda x: product_name in list(x)
        )
    ]

    return recommendations[['consequents',
                            'confidence',
                            'lift']]

print(recommend_products('WHITE HANGING HEART T-LIGHT HOLDER'))



# Analysis and Conclusions

- Association Rules help identify product relationships.
- Higher confidence means stronger association.
- Lift greater than 1 indicates positive product relationships.
- Threshold experimentation changes the number of rules generated.
- Market basket analysis improves recommendation systems.


# Interview Questions

## 1. What is Lift and Why is it Important?
- Lift measures how strongly two items are associated compared to random occurrence.
- Lift > 1 indicates positive association between products.

## 2. What are Support and Confidence?
- Support measures how frequently an itemset appears in the dataset.
- Confidence measures the likelihood of purchasing one item given another item is purchased.

### Formulas:
- Support(A) = Transactions containing A / Total Transactions
- Confidence(A→B) = Support(A and B) / Support(A)

## 3. Limitations of Association Rule Mining
- Computationally expensive for large datasets.
- Generates too many rules.
- May identify misleading associations.
- Requires careful threshold tuning.
